# Kenya - LLM Exploratory Analysis

This notebook is ready for Google Colab. Upload the two Excel files into Colab's `/content` folder, or mount Google Drive and set the file paths in the setup cell.

**Expected input filenames if uploaded to `/content`**
- `Kenya Audience Analysis Comments.xlsx`
- `Kenya Content Analysis Snippets.xlsx`

**Main outputs**
- `Kenya - LLM Exploratory Data Analyses.xlsx`
- `Kenya - LLM Exploratory Report.md`
- `kenya_audience_row_coding.csv`
- `kenya_content_row_coding.csv`

The notebook caches OpenAI responses in `llm_cache.jsonl`, so if a run stops midway, re-running will skip rows already coded.

## 0 - Colab setup and OpenAI key

**In Colab:**
1. Upload both Excel files to `/content`, or mount Drive and update `AUDIENCE_XLSX` / `CONTENT_XLSX` below.
2. Preferred key setup: add a Colab Secret named `OPENAI_API_KEY` from the key icon in the left sidebar, then enable notebook access.
3. If the secret is not available, the setup cell will ask for the key with a hidden prompt.

Do **not** paste your real key into a saved/shared notebook cell.

In [ ]:
# Run this first in Colab. It is harmless locally if packages already exist.
import importlib.util
import subprocess
import sys

for package, import_name in [("openai", "openai"), ("openpyxl", "openpyxl")]:
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("Dependencies ready.")

In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import os
import re
import textwrap
import time
from collections import Counter, defaultdict
from datetime import datetime
from getpass import getpass
from pathlib import Path

from openai import OpenAI
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB and not os.getenv("OPENAI_API_KEY"):
    try:
        from google.colab import userdata  # type: ignore
        secret_key = userdata.get("OPENAI_API_KEY")
        if secret_key:
            os.environ["OPENAI_API_KEY"] = secret_key
            print("Loaded OPENAI_API_KEY from Colab Secrets.")
    except Exception:
        pass

if not os.getenv("OPENAI_API_KEY"):
    entered_key = getpass("OpenAI API key (input hidden): ").strip()
    if entered_key:
        os.environ["OPENAI_API_KEY"] = entered_key

MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
COUNTRY = "Kenya"

if IN_COLAB:
    # Default Colab workflow: upload the Excel files into /content.
    BASE_DIR = Path("/content")
    AUDIENCE_XLSX = Path(os.getenv("AUDIENCE_XLSX", "/content/Kenya Audience Analysis Comments.xlsx"))
    CONTENT_XLSX = Path(os.getenv("CONTENT_XLSX", "/content/Kenya Content Analysis Snippets.xlsx"))
    OUT_DIR = Path(os.getenv("OUT_DIR", "/content/kenya_llm_exploratory_outputs"))
else:
    # Local Mac workflow used when this notebook is run outside Colab.
    BASE_DIR = Path("/Users/dhruvbhanderi/Documents/USC/New Research Engineer")
    AUDIENCE_XLSX = Path(os.getenv("AUDIENCE_XLSX", str(BASE_DIR / "Final datasets" / "Kenya Audience Analysis Comments.xlsx")))
    CONTENT_XLSX = Path(os.getenv("CONTENT_XLSX", str(BASE_DIR / "Final datasets" / "Kenya Content Analysis Snippets.xlsx")))
    OUT_DIR = Path(os.getenv("OUT_DIR", str(BASE_DIR / "Codex" / "outputs" / "kenya_llm_exploratory_from_attached")))

OUT_DIR.mkdir(parents=True, exist_ok=True)

CACHE_JSONL = OUT_DIR / "llm_cache.jsonl"
OUT_WORKBOOK = OUT_DIR / "Kenya - LLM Exploratory Data Analyses.xlsx"
OUT_REPORT = OUT_DIR / "Kenya - LLM Exploratory Report.md"
OUT_AUDIENCE_CSV = OUT_DIR / "kenya_audience_row_coding.csv"
OUT_CONTENT_CSV = OUT_DIR / "kenya_content_row_coding.csv"

# Set SAMPLE_MODE=True for a quick smoke test. Set False for the final full run.
SAMPLE_MODE = False
SAMPLE_N_PER_DATASET = 20

# Keep rows from overwhelming the model. Longer texts are clipped for coding, but full text is still exported.
MAX_TEXT_CHARS_FOR_LLM = 3500
REQUEST_SLEEP_SECONDS = 0.05

missing = [str(path) for path in [AUDIENCE_XLSX, CONTENT_XLSX] if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Could not find input file(s):\n- "
        + "\n- ".join(missing)
        + "\n\nIn Colab, upload the two Excel files to /content, or mount Drive and set "
        + "AUDIENCE_XLSX and CONTENT_XLSX to the Drive paths."
    )

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is not set.")

client = OpenAI()
print("running in Colab:", IN_COLAB)
print("model:", MODEL)
print("audience input:", AUDIENCE_XLSX)
print("content input:", CONTENT_XLSX)
print("output folder:", OUT_DIR)
print("sample mode:", SAMPLE_MODE, SAMPLE_N_PER_DATASET if SAMPLE_MODE else "all rows")

## 1 - Load and normalize the Excel files

In [ ]:
CREATOR_MAP = {
    "Andrew": "Andrew Kibe",
    "Andrew Kibe": "Andrew Kibe",
    "EricA": "Eric Amunga (Amerix)",
    "Eric (Amerix)": "Eric Amunga (Amerix)",
    "Eric Amunga / Amerix": "Eric Amunga (Amerix)",
    "Eric Amunga (Amerix)": "Eric Amunga (Amerix)",
    "Rixpoet": "Onyango Otieno (Rixpoet)",
    "Onyango Otieno (Rixpoet)": "Onyango Otieno (Rixpoet)",
    "Eddy": "Eddy Kimani",
    "Eddy Kimani": "Eddy Kimani",
    "Philip Karanja": "Philip Karanja",
}

ORIENTATION = {
    "Andrew Kibe": "regressive / manosphere-adjacent",
    "Eric Amunga (Amerix)": "regressive / manosphere-adjacent",
    "Eddy Kimani": "progressive / vulnerability-forward",
    "Onyango Otieno (Rixpoet)": "progressive / vulnerability-forward",
    "Philip Karanja": "progressive / vulnerability-forward",
}

AUDIENCE_SUMMARY_SHEETS = {"summary", "summary metrics"}
CONTENT_SUMMARY_SHEETS = {"summary", "summary metrics"}


def clean_value(value):
    if value is None:
        return ""
    value = str(value).strip()
    value = re.sub(r"\s+", " ", value)
    return value


def row_dicts_from_sheet(ws):
    rows = list(ws.iter_rows(values_only=True))
    if not rows:
        return []
    headers = [clean_value(h) for h in rows[0]]
    out = []
    for raw in rows[1:]:
        item = {headers[i]: clean_value(raw[i]) if i < len(raw) else "" for i in range(len(headers)) if headers[i]}
        if any(item.values()):
            out.append(item)
    return out


def canonical_creator(raw, fallback_sheet=""):
    raw = clean_value(raw)
    if raw in CREATOR_MAP:
        return CREATOR_MAP[raw]
    for key, value in CREATOR_MAP.items():
        if key and key.lower() in raw.lower():
            return value
    sheet = fallback_sheet.lower()
    if "andrew" in sheet or "kibe" in sheet:
        return "Andrew Kibe"
    if "eric" in sheet or "amerix" in sheet:
        return "Eric Amunga (Amerix)"
    if "eddy" in sheet:
        return "Eddy Kimani"
    if "rixpoet" in sheet or "onyango" in sheet:
        return "Onyango Otieno (Rixpoet)"
    if "philip" in sheet:
        return "Philip Karanja"
    return raw or fallback_sheet


def extract_url(text):
    match = re.search(r"https?://\S+", clean_value(text))
    return match.group(0) if match else ""


def load_audience(path):
    wb = load_workbook(path, read_only=True, data_only=True)
    rows = []
    for ws in wb.worksheets:
        if ws.title.strip().lower() in AUDIENCE_SUMMARY_SHEETS:
            continue
        for item in row_dicts_from_sheet(ws):
            comment = clean_value(item.get("Comment"))
            if not comment:
                continue
            creator = canonical_creator(item.get("Influencer"), ws.title)
            rows.append({
                "dataset": "audience",
                "country": COUNTRY,
                "row_id": clean_value(item.get("Comment ID")),
                "source_sheet": ws.title,
                "creator": creator,
                "orientation": ORIENTATION.get(creator, "unknown"),
                "platform": clean_value(item.get("Platform")),
                "source_url": clean_value(item.get("Source URL")),
                "content_type": "Audience comment",
                "context": "",
                "text": comment,
            })
    return rows


def load_content(path):
    wb = load_workbook(path, read_only=True, data_only=True)
    rows = []
    for ws in wb.worksheets:
        if ws.title.strip().lower() in CONTENT_SUMMARY_SHEETS:
            continue
        for item in row_dicts_from_sheet(ws):
            text = clean_value(item.get("Text"))
            if not text:
                continue
            creator = canonical_creator(item.get("Influencer"), ws.title)
            context = clean_value(item.get("Context"))
            rows.append({
                "dataset": "content",
                "country": COUNTRY,
                "row_id": clean_value(item.get("Segment ID") or item.get("Tweet ID")),
                "source_sheet": ws.title,
                "creator": creator,
                "orientation": ORIENTATION.get(creator, "unknown"),
                "platform": clean_value(item.get("Platform")),
                "source_url": extract_url(context),
                "content_type": clean_value(item.get("Content Type")),
                "context": context,
                "text": text,
            })
    return rows


audience_rows = load_audience(AUDIENCE_XLSX)
content_rows = load_content(CONTENT_XLSX)

if SAMPLE_MODE:
    audience_rows = audience_rows[:SAMPLE_N_PER_DATASET]
    content_rows = content_rows[:SAMPLE_N_PER_DATASET]

print(f"audience rows: {len(audience_rows):,}")
print(f"content rows:  {len(content_rows):,}")
print("audience by creator:", dict(Counter(r["creator"] for r in audience_rows)))
print("content by creator:", dict(Counter(r["creator"] for r in content_rows)))
print("first audience row:", audience_rows[0])
print("first content row:", content_rows[0])

## 2 - QA checks

In [ ]:
def qa_rows(rows, name):
    checks = []
    checks.append(("has rows", len(rows) > 0, len(rows)))
    checks.append(("row_id populated", all(r["row_id"] for r in rows), sum(1 for r in rows if not r["row_id"])))
    checks.append(("text populated", all(r["text"] for r in rows), sum(1 for r in rows if not r["text"])))
    checks.append(("creator populated", all(r["creator"] for r in rows), sum(1 for r in rows if not r["creator"])))
    checks.append(("orientation mapped", all(r["orientation"] != "unknown" for r in rows), sum(1 for r in rows if r["orientation"] == "unknown")))
    checks.append(("reasonable text length", all(0 < len(r["text"]) <= 20000 for r in rows), max(len(r["text"]) for r in rows)))
    print(f"\n{name}")
    for label, ok, detail in checks:
        print(("PASS" if ok else "FAIL"), "-", label, "-", detail)
    return all(ok for _, ok, _ in checks)

qa_ok = qa_rows(audience_rows, "Audience QA") and qa_rows(content_rows, "Content QA")
if not qa_ok:
    raise AssertionError("Fix QA failures before running LLM coding.")

## 3 - LLM coding with cache

Each row is coded into a compact JSON object. The same cache is reused across runs using dataset, row id, model, and a text hash.

In [ ]:
ALLOWED_THEMES = [
    "relationships/love/loyalty",
    "gender roles and expectations",
    "male self-improvement/discipline/purpose",
    "money/status/provision",
    "mental health/trauma/healing",
    "violence/abuse/accountability",
    "family/fatherhood/parenting",
    "religion/morality/spirituality",
    "women critique/misogynistic framing",
    "equality/reciprocity/mutual respect",
    "men withdrawing/peace over performance",
    "humor/sarcasm/low-substance uptake",
    "other/unclear",
]

SYSTEM_PROMPT = """You are a careful qualitative research assistant for a media and masculinity project.
Code each row conservatively using only the supplied text and context. Return valid JSON only.
Do not invent facts, names, or context beyond the row. Use lowercase labels where possible.
"""

AUDIENCE_TASK = """Code this audience comment.
Return a JSON object with exactly these keys:
- themes: array of 1-4 labels from the allowed theme list
- primary_theme: one label from the allowed theme list
- sentiment: one of positive, neutral, negative, mixed
- emotions: array of 0-3 labels
- stance: one of supports creator/message, critiques creator/message, mixed/ambivalent, personal testimony, joking/low-substance, unclear
- toxicity_0_3: integer 0-3
- misogyny_0_3: integer 0-3
- hate_speech: one of none, possible, clear
- women_portrayal: one of not present, positive/equal/respected, neutral/mixed, negative/problematic
- moral_foundations: array using any of care/harm, fairness/cheating, loyalty/betrayal, authority/subversion, sanctity/degradation, liberty/oppression, none/unclear
- key_terms: array of 0-6 short terms
- rationale: one short sentence explaining the coding
"""

CONTENT_TASK = """Code this creator content snippet.
Return a JSON object with exactly these keys:
- themes: array of 1-4 labels from the allowed theme list
- primary_theme: one label from the allowed theme list
- sentiment: one of positive, neutral, negative, mixed
- emotions: array of 0-3 labels
- framing: one of traditional masculinity, vulnerability/healing, provider/status, anti-women/manosphere, equality/reciprocity, family/fatherhood, motivational/self-help, unclear/mixed
- argument_type: one of personal story, advice/prescription, moral claim, social critique, joke/sarcasm, informational, unclear
- toxicity_0_3: integer 0-3
- misogyny_0_3: integer 0-3
- hate_speech: one of none, possible, clear
- women_portrayal: one of not present, positive/equal/respected, neutral/mixed, negative/problematic
- moral_foundations: array using any of care/harm, fairness/cheating, loyalty/betrayal, authority/subversion, sanctity/degradation, liberty/oppression, none/unclear
- key_terms: array of 0-6 short terms
- rationale: one short sentence explaining the coding
"""

DEFAULT_CODE = {
    "themes": ["other/unclear"],
    "primary_theme": "other/unclear",
    "sentiment": "neutral",
    "emotions": [],
    "toxicity_0_3": 0,
    "misogyny_0_3": 0,
    "hate_speech": "none",
    "women_portrayal": "not present",
    "moral_foundations": ["none/unclear"],
    "key_terms": [],
    "rationale": "No clear signal beyond the text provided.",
}


def text_hash(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]


def cache_key(row):
    return f"{MODEL}|{row['dataset']}|{row['row_id']}|{text_hash(row['text'])}"


def load_cache(path):
    cache = {}
    if not path.exists():
        return cache
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            item = json.loads(line)
            cache[item["key"]] = item["result"]
    return cache


def append_cache(path, key, result):
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps({"key": key, "result": result}, ensure_ascii=False) + "\n")


def normalize_code(result, dataset):
    out = dict(DEFAULT_CODE)
    out.update(result or {})
    if dataset == "audience":
        out.setdefault("stance", "unclear")
    else:
        out.setdefault("framing", "unclear/mixed")
        out.setdefault("argument_type", "unclear")
    if isinstance(out.get("themes"), str):
        out["themes"] = [out["themes"]]
    out["themes"] = [t for t in out.get("themes", []) if t] or ["other/unclear"]
    if out.get("primary_theme") not in ALLOWED_THEMES:
        out["primary_theme"] = out["themes"][0] if out["themes"] else "other/unclear"
    for field in ["toxicity_0_3", "misogyny_0_3"]:
        try:
            out[field] = max(0, min(3, int(out.get(field, 0))))
        except Exception:
            out[field] = 0
    for field in ["emotions", "moral_foundations", "key_terms"]:
        if isinstance(out.get(field), str):
            out[field] = [out[field]] if out[field] else []
        if not isinstance(out.get(field), list):
            out[field] = []
    return out


def extract_json(text):
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?", "", text).strip()
        text = re.sub(r"```$", "", text).strip()
    start = text.find("{")
    end = text.rfind("}")
    if start >= 0 and end >= start:
        text = text[start:end + 1]
    return json.loads(text)


def code_row(row, cache):
    key = cache_key(row)
    if key in cache:
        return cache[key]

    task = AUDIENCE_TASK if row["dataset"] == "audience" else CONTENT_TASK
    text_for_llm = row["text"][:MAX_TEXT_CHARS_FOR_LLM]
    user_prompt = f"""
Allowed themes: {json.dumps(ALLOWED_THEMES)}
Dataset: {row['dataset']}
Creator: {row['creator']}
Orientation: {row['orientation']}
Platform: {row['platform']}
Content type: {row['content_type']}
Context: {row['context'][:800]}

{task}

Text:
{text_for_llm}
""".strip()

    last_error = None
    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                ],
                response_format={"type": "json_object"},
                temperature=0,
            )
            raw = response.choices[0].message.content or "{}"
            result = normalize_code(extract_json(raw), row["dataset"])
            append_cache(CACHE_JSONL, key, result)
            cache[key] = result
            time.sleep(REQUEST_SLEEP_SECONDS)
            return result
        except Exception as exc:
            last_error = exc
            time.sleep(2 ** attempt)
    raise RuntimeError(f"Failed to code {row['dataset']} row {row['row_id']}: {last_error}")

cache = load_cache(CACHE_JSONL)
print(f"loaded cached rows: {len(cache):,}")


In [ ]:
def code_dataset(rows, name):
    coded = []
    total = len(rows)
    for i, row in enumerate(rows, start=1):
        result = code_row(row, cache)
        coded.append({**row, **result})
        if i == 1 or i % 25 == 0 or i == total:
            print(f"{name}: coded {i:,}/{total:,}")
    return coded

coded_audience = code_dataset(audience_rows, "audience")
coded_content = code_dataset(content_rows, "content")

print("done")

## 4 - Summaries

In [ ]:
def count_field(rows, field):
    return Counter(clean_value(r.get(field)) for r in rows if clean_value(r.get(field)))


def count_list_field(rows, field):
    counts = Counter()
    for row in rows:
        value = row.get(field, [])
        if isinstance(value, str):
            value = [value]
        counts.update(v for v in value if v)
    return counts


def cross_tab(rows, row_field, col_field):
    row_values = sorted({clean_value(r.get(row_field)) for r in rows if clean_value(r.get(row_field))})
    col_values = sorted({clean_value(r.get(col_field)) for r in rows if clean_value(r.get(col_field))})
    table = [[row_field, *col_values]]
    for rv in row_values:
        table.append([rv, *[sum(1 for r in rows if clean_value(r.get(row_field)) == rv and clean_value(r.get(col_field)) == cv) for cv in col_values]])
    return table


def top_examples(rows, n=20):
    ranked = sorted(
        rows,
        key=lambda r: (int(r.get("toxicity_0_3", 0)) + int(r.get("misogyny_0_3", 0)), len(r.get("text", ""))),
        reverse=True,
    )
    return ranked[:n]

summary = {
    "audience_sentiment": count_field(coded_audience, "sentiment"),
    "content_sentiment": count_field(coded_content, "sentiment"),
    "audience_themes": count_list_field(coded_audience, "themes"),
    "content_themes": count_list_field(coded_content, "themes"),
    "audience_stance": count_field(coded_audience, "stance"),
    "content_framing": count_field(coded_content, "framing"),
}

for name, counts in summary.items():
    print("\n", name)
    for label, n in counts.most_common(12):
        print(f"  {label}: {n}")


## 5 - Export workbook, CSVs, and report

In [ ]:
BASE_COLUMNS = [
    "dataset", "country", "row_id", "source_sheet", "creator", "orientation", "platform",
    "source_url", "content_type", "context", "text",
]
AUDIENCE_CODE_COLUMNS = [
    "themes", "primary_theme", "sentiment", "emotions", "stance", "toxicity_0_3",
    "misogyny_0_3", "hate_speech", "women_portrayal", "moral_foundations", "key_terms", "rationale",
]
CONTENT_CODE_COLUMNS = [
    "themes", "primary_theme", "sentiment", "emotions", "framing", "argument_type", "toxicity_0_3",
    "misogyny_0_3", "hate_speech", "women_portrayal", "moral_foundations", "key_terms", "rationale",
]


def excel_value(value):
    if isinstance(value, (list, dict)):
        return json.dumps(value, ensure_ascii=False)
    return value


def write_csv(path, rows, columns):
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        for row in rows:
            writer.writerow({c: excel_value(row.get(c, "")) for c in columns})


def add_table_sheet(wb, title, headers, rows):
    ws = wb.create_sheet(title)
    ws.append(headers)
    for row in rows:
        ws.append([excel_value(row.get(h, "")) for h in headers])
    if ws.max_row > 1 and ws.max_column > 1:
        table_ref = f"A1:{get_column_letter(ws.max_column)}{ws.max_row}"
        tab = Table(displayName=re.sub(r"[^A-Za-z0-9_]", "_", title)[:30], ref=table_ref)
        tab.tableStyleInfo = TableStyleInfo(name="TableStyleMedium2", showFirstColumn=False, showLastColumn=False, showRowStripes=True, showColumnStripes=False)
        ws.add_table(tab)
    ws.freeze_panes = "A2"
    for cell in ws[1]:
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = PatternFill("solid", fgColor="214E5F")
        cell.alignment = Alignment(wrap_text=True, vertical="top")
    widths = {
        "A": 16, "B": 12, "C": 22, "D": 28, "E": 28, "F": 30, "G": 16,
        "H": 34, "I": 24, "J": 42, "K": 70, "L": 40, "M": 28, "N": 18,
    }
    for col, width in widths.items():
        ws.column_dimensions[col].width = width
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(wrap_text=True, vertical="top")


def add_matrix(ws, start_row, start_col, matrix):
    for r, row in enumerate(matrix, start=start_row):
        for c, value in enumerate(row, start=start_col):
            ws.cell(r, c, value)


def counts_matrix(title, counts):
    return [[title, "count"], *[[k, v] for k, v in counts.most_common()]]


def write_outputs():
    aud_cols = BASE_COLUMNS + AUDIENCE_CODE_COLUMNS
    con_cols = BASE_COLUMNS + CONTENT_CODE_COLUMNS
    write_csv(OUT_AUDIENCE_CSV, coded_audience, aud_cols)
    write_csv(OUT_CONTENT_CSV, coded_content, con_cols)

    wb = Workbook()
    ws = wb.active
    ws.title = "Summary"
    ws["A1"] = "Kenya - LLM Exploratory Data Analyses"
    ws["A1"].font = Font(size=16, bold=True, color="163A2A")
    ws["A2"] = f"Generated {datetime.now().strftime('%Y-%m-%d %H:%M')} with model {MODEL}"
    ws["A4"] = "Input audience workbook"
    ws["B4"] = str(AUDIENCE_XLSX)
    ws["A5"] = "Input content workbook"
    ws["B5"] = str(CONTENT_XLSX)
    ws["A6"] = "Audience rows"
    ws["B6"] = len(coded_audience)
    ws["A7"] = "Content rows"
    ws["B7"] = len(coded_content)
    ws["A8"] = "Cache file"
    ws["B8"] = str(CACHE_JSONL)

    add_matrix(ws, 11, 1, counts_matrix("Audience sentiment", summary["audience_sentiment"]))
    add_matrix(ws, 11, 4, counts_matrix("Content sentiment", summary["content_sentiment"]))
    add_matrix(ws, 11, 7, counts_matrix("Audience stance", summary["audience_stance"]))
    add_matrix(ws, 11, 10, counts_matrix("Content framing", summary["content_framing"]))
    add_matrix(ws, 28, 1, counts_matrix("Audience themes", summary["audience_themes"]))
    add_matrix(ws, 28, 4, counts_matrix("Content themes", summary["content_themes"]))
    add_matrix(ws, 50, 1, cross_tab(coded_audience, "creator", "sentiment"))
    add_matrix(ws, 50, 8, cross_tab(coded_content, "creator", "sentiment"))
    for row in ws.iter_rows():
        for cell in row:
            cell.alignment = Alignment(wrap_text=True, vertical="top")
    for col in range(1, 13):
        ws.column_dimensions[get_column_letter(col)].width = 24

    add_table_sheet(wb, "audience_rows", aud_cols, coded_audience)
    add_table_sheet(wb, "content_rows", con_cols, coded_content)

    quote_headers = ["dataset", "row_id", "creator", "primary_theme", "sentiment", "toxicity_0_3", "misogyny_0_3", "text", "rationale"]
    quote_rows = top_examples(coded_audience, 15) + top_examples(coded_content, 15)
    add_table_sheet(wb, "high_signal_examples", quote_headers, quote_rows)

    wb.save(OUT_WORKBOOK)

    report = []
    report.append("# Kenya - LLM Exploratory Report")
    report.append("")
    report.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    report.append(f"Model: `{MODEL}`")
    report.append("")
    report.append(f"Audience rows coded: {len(coded_audience):,}")
    report.append(f"Content rows coded: {len(coded_content):,}")
    report.append("")
    for label, counts in summary.items():
        report.append(f"## {label.replace('_', ' ').title()}")
        for key, value in counts.most_common(12):
            report.append(f"- {key}: {value}")
        report.append("")
    OUT_REPORT.write_text("\n".join(report), encoding="utf-8")

write_outputs()
print("wrote", OUT_WORKBOOK)
print("wrote", OUT_REPORT)
print("wrote", OUT_AUDIENCE_CSV)
print("wrote", OUT_CONTENT_CSV)


## 6 - Quick preview

In [ ]:
print(OUT_REPORT.read_text(encoding="utf-8")[:3000])

## 7 - Optional Colab download

After the export cell finishes, run this cell in Colab to download the workbook, report, and CSV files.

In [ ]:
if IN_COLAB:
    from google.colab import files  # type: ignore
    for output_path in [OUT_WORKBOOK, OUT_REPORT, OUT_AUDIENCE_CSV, OUT_CONTENT_CSV]:
        if output_path.exists():
            files.download(str(output_path))
else:
    print("Outputs are already on disk at:", OUT_DIR)

## Notes

- In Colab, the OpenAI key is read from a Colab Secret named `OPENAI_API_KEY`, or from the hidden prompt in the setup cell.
- If files are uploaded through the Colab Files panel, keep the default paths: `/content/Kenya Audience Analysis Comments.xlsx` and `/content/Kenya Content Analysis Snippets.xlsx`.
- If files are in Google Drive, mount Drive and update `AUDIENCE_XLSX`, `CONTENT_XLSX`, and optionally `OUT_DIR` in the setup cell.
- To use a different model, set `OPENAI_MODEL` before the setup cell or change `MODEL` there.
- `SAMPLE_MODE=True` runs a small smoke test; `SAMPLE_MODE=False` runs all rows.
- Re-running is resumable because completed row codings are stored in `llm_cache.jsonl`.
- If you want to force a fresh LLM run, delete `llm_cache.jsonl` in the output folder.